# Federated Analytics

This notebook serves as a base example to how to explore on datasets by calculating some different analytics functions, without directly initializing a model and a training. We will see how the mean is calculated on nodes which contain the ADNI Dataset.

### Setting the node up
Before running this notebook we configure 2 nodes: <br/>
* **Node 1 :**
  * Add a dataset `fedbiomed node -p my-first-node dataset add`
  * Select option, choose the name, tags and description of the dataset
  * Check that your data has been added by executing `fedbiomed node -p my-first-node list`
  * Run the node using `fedbiomed node -p my-first-node start`. <br/>

* **Node 2 :**
  * Repeat the same steps for `my-second-node`

Wait until you get `Starting task manager`, it means node is online.

### Setting up the experiment

We set up an experiment, in order to select the nodes which contain the datasets and perform federated analytics on them.

## Tabular Dataset

For a tabular dataset you can input which columns you want to compute the analytics on. The columns are given as a list, which can be a list of column names or column indexes.

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment

# The `Experiment` scans all connected nodes for datasets matching the `'tabular'` tag. 
# Two nodes are discovered and registered for the experiment.
tags = ['tabular']
exp = Experiment(tags=tags)

In [ ]:
# Both nodes host CSV datasets with the same 7 columns and compatible types.
# Node 1 has 10 668 rows, Node 2 has 17 965 rows. 
# This metadata is returned by nodes without exposing any raw data.
exp.training_data().data()

In [ ]:
# `fetch_stats('mean')` sends an analytics request to both nodes. 
# Each node computes the mean and count for every numeric column locally and returns them.
# Results are stored in an `FAResult` object and cached.
result = exp.analytics.fetch_stats("mean")

In [ ]:
result.node_ids

In [ ]:
result.schema

In [ ]:
result.node_stats()

In [ ]:
# `available_stats()` lists statistics present in the node replies (`count`, `mean`).
# `computable_stats()` additionally includes `sum`, which can be derived locally from `mean × count` without re-querying nodes.
print(result.available_stats())
print(result.computable_stats())

In [ ]:
# `global_stats('mean')` combines per-node means into a single count-weighted average across all nodes.
result.global_stats("mean")

In [ ]:
# `exp.analytics.mean()` is a shorthand that calls `fetch_stats('mean')` then `global_stats('mean')`. 
# The log confirms the result is served from cache — nodes are not contacted again.
exp.analytics.mean()

In [ ]:
# `min` and `max` are not recognized statistics. 
# `fetch_stats` raises a `FedbiomedError` immediately, before contacting any node. 
# The previously fetched `result` object is unaffected and still holds the mean data.
result = exp.analytics.fetch_stats(["min", "max"])

In [ ]:
result.node_stats()

In [ ]:
# `'histogram'` passes client-side validation but is rejected by both nodes, which do not yet implement it.
# The error is fatal — no partial results are saved and the request does not overwrite the cached result.
result = exp.analytics.fetch_stats("histogram")

In [ ]:
result = exp.analytics.fetch_stats_with_args(stats_args={
    "price": {
        "histogram": {
            "bin_edges": [100,125,150,175,200,250]
        }
    }
})

In [ ]:
# After two failed histogram requests, `result` still holds the mean data from the original successful call.
# Failed requests never overwrite the cached result.
result.node_stats()

## Mnist Dataset

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment

tags = ['#MNIST', '#dataset']
exp = Experiment(tags=tags)
exp.training_data().data()

In [ ]:
result = exp.analytics.fetch_stats("mean")

In [ ]:
result.node_ids

In [ ]:
# The schema is empty because MNIST has no named tabular columns. 
# With no columns to operate on, `available_stats`, `computable_stats`, and `node_stats` are all empty. 
# No error is raised — the request simply returns nothing.
print(result.schema)
print(result.available_stats())
print(result.computable_stats())

In [ ]:
result.node_stats()

## Medical Folder Dataset

The IXI dataset combines NIfTI image modalities (`T1`, `T2`, `label`) with a demographics CSV. Analytics can only be computed on named tabular columns; image modalities appear in the schema but produce no statistics.

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment

tags = ['ixi']
exp = Experiment(tags=tags)
exp.training_data().data()

In [ ]:
result = exp.analytics.fetch_stats("mean")

In [ ]:
result.node_ids

In [ ]:
# The schema reflects the nested structure: `demographics` has named sub-columns (e.g. `AGE`, `HEIGHT`),
# while `T1`, `T2`, and `label` are image modalities with no sub-columns.
result.schema

In [ ]:
result.available_stats()

In [ ]:
result.node_stats()

Only numeric demographics columns have valid stats. `DOB`, `STUDY_DATE`, and `SITE_NAME` return `nan` with `count: 0` because they are non-numeric (dates or text). Image modality keys (`T1`, `T2`, `label`) are empty — per-voxel statistics are not computed.